# Reconciling Campaign Runs onto the Calendar

This notebook demonstrates `solsys_code/campaign_reconciler.py`,
`solsys_code/allocation_projector.py` and
`solsys_code/management/commands/reconcile_campaign_runs.py` /
`cutover_classical_allocations.py` (Phase 29 "the reconciler", extended by Phase 35's
allocation layer and classical cutover).

It demonstrates, against a throwaway scratch copy of the developer database:

- **The one-time classical cutover (Phase 35, D-15..D-18)** — the before-and-after diff
  ROADMAP Success Criterion 5 asks for: converting the legacy blank-url classical
  `CalendarEvent`s and the retired `RUN:{pk}:{date}` per-night family into the current
  `ALLOC:` allocation layer, with the D-18 unexplainable-event report rendered as real
  output rather than hidden, and four end-state properties asserted in code
- `reconcile_run()`'s four current dispatch branches (D-09/D-10): a class-wide allocation, a
  satellite run, a queue-sourced run (which now always gets a whole-window container,
  regardless of its site), and a campaign-less classical run (which gets one `ALLOC:`
  allocation night per observing night)
- A `--dry-run` sweep via `call_command`, showing the `would_create` counters and that no
  `CalendarEvent` rows are written
- A real sweep, then idempotency: a second real sweep reports `created: 0, updated: 0`
- A real sweep touching nothing outside the `RUN:`/`ALLOC:` namespaces it owns — never a
  facility-url-keyed observation event
- The D-05/D-07 observation handoff: linking a real, placed `ObservationRecord` retires an
  allocation night; unlinking restores it

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records, runs the cutover command, and creates
`CampaignRun`/`CalendarEvent`/`ObservationRecord` rows) and is therefore **NOT** run during
Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` -- three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

Before `django.setup()` runs, the setup cell below also copies the developer database
(`src/fomo_db.sqlite3`) to a throwaway scratch file and points `FOMO_DATABASE_PATH` at
that copy, so this notebook's entire run -- every row it creates, converts or removes --
happens against a copy that is torn down at the end, never against the developer database
itself (UAT G-33-4). The scratch copy is then migrated to head, since the developer
database has not yet had migration 0018 (`CampaignRun.night_start_utc`/`night_end_utc`,
plan 35-03) applied and the cutover command's own field reads depend on it.


In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

# Copy the developer database to a scratch file and point FOMO_DATABASE_PATH at the
# copy BEFORE django.setup() -- that call is what materialises DATABASES, so the
# variable must already be exported. This is what makes the developer database
# read-only for this notebook's entire run (UAT G-33-4).
import shutil
import tempfile

dev_db_path = repo_root_path / 'src' / 'fomo_db.sqlite3'
if not dev_db_path.exists():
    raise RuntimeError(f'No developer database at {dev_db_path}; run `python manage.py migrate` first.')
scratch_db_dir = Path(tempfile.mkdtemp(prefix='fomo-notebook-db-'))
scratch_db_path = scratch_db_dir / 'fomo_db.sqlite3'
shutil.copy2(dev_db_path, scratch_db_path)
os.environ['FOMO_DATABASE_PATH'] = str(scratch_db_path)

django.setup()

# This notebook intentionally imports only the campaign-coordination models and the
# reconciler below -- never the ephemeris view/computation modules, which trigger a large
# one-time SPICE kernel download on first import.

from django.conf import settings as django_settings

resolved_db_name = django_settings.DATABASES['default']['NAME']
assert resolved_db_name == str(scratch_db_path), (
    'This notebook must never write to the developer database -- resolved DB '
    f'{resolved_db_name!r} is not the scratch copy {str(scratch_db_path)!r}.'
)

# The scratch copy is cloned from the real developer database, which is generally
# behind head -- migrate the copy to whatever head currently is (as of Phase 35 plan
# 35-21, that includes migration 0020) before touching CampaignRun.
from django.core.management import call_command as _call_command

_call_command('migrate', verbosity=0)

print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')
print(f'Resolved database: {resolved_db_name!r} (scratch copy of the developer database, migrated to head)')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel'
Resolved database: '/tmp/fomo-notebook-db-ycs_q23l/fomo_db.sqlite3' (scratch copy of the developer database, migrated to head)


## The classical cutover: before-and-after diff

Everything in this section runs against the SAME scratch copy of the developer database
created in the setup cell above, captured BEFORE any of this notebook's own fixtures are
seeded further down -- this is the pristine pre-cutover state plan 35-06's own
real-database proof measured, so the before/after numbers below are the same worked
example the operator runbook quotes.

In [2]:
import io

from django.core.management import call_command
from django.core.management.base import CommandError
from tom_calendar.models import CalendarEvent

from solsys_code.allocation_projector import ALLOC_URL_NAMESPACE
from solsys_code.campaign_reconciler import RUN_URL_NAMESPACE
from solsys_code.models import CampaignRun


def key_family_counts():
    """Classifies every CalendarEvent by key family, matching plan 35-06's own
    real-database proof: RUN:{pk}:{date} date-bearing nights, bare RUN:{pk} containers,
    blank-url legacy classical events, ALLOC:-keyed allocation nights, and everything else
    -- facility-url-keyed observation events."""
    urls = list(CalendarEvent.objects.values_list('url', flat=True))
    run_dated = sum(1 for u in urls if u.startswith(RUN_URL_NAMESPACE) and ':' in u[len(RUN_URL_NAMESPACE) :])
    run_bare = sum(1 for u in urls if u.startswith(RUN_URL_NAMESPACE) and ':' not in u[len(RUN_URL_NAMESPACE) :])
    blank_url = sum(1 for u in urls if u == '')
    alloc = sum(1 for u in urls if u.startswith(ALLOC_URL_NAMESPACE))
    total = len(urls)
    facility_url = total - run_dated - run_bare - blank_url - alloc
    return {
        'total': total,
        'run_dated (RUN:{pk}:{date})': run_dated,
        'run_bare (RUN:{pk})': run_bare,
        'blank_url': blank_url,
        'alloc (ALLOC:)': alloc,
        'facility_url': facility_url,
    }


def facility_url_snapshot():
    """(pk -> url, title, description, start_time, end_time, modified) for every
    facility-url-keyed observation event -- the set the cutover must leave byte-identical."""
    qs = (
        CalendarEvent.objects.exclude(url__startswith=RUN_URL_NAMESPACE)
        .exclude(url__startswith=ALLOC_URL_NAMESPACE)
        .exclude(url='')
    )
    return {ev.pk: (ev.url, ev.title, ev.description, ev.start_time, ev.end_time, ev.modified) for ev in qs}


before_counts = key_family_counts()
before_campaign_run_count = CampaignRun.objects.count()
before_facility_snapshot = facility_url_snapshot()

print('=== BEFORE the cutover ===')
for key, value in before_counts.items():
    print(f'  {key:28}: {value}')
print(f'  {"CampaignRun rows":28}: {before_campaign_run_count}')
print(f'Facility-url events snapshotted: {len(before_facility_snapshot)}')

=== BEFORE the cutover ===
  total                       : 233
  run_dated (RUN:{pk}:{date}) : 0
  run_bare (RUN:{pk})         : 16
  blank_url                   : 10
  alloc (ALLOC:)              : 48
  facility_url                : 159
  CampaignRun rows            : 45
Facility-url events snapshotted: 159


Both cells below are expected to raise. On this database the cutover finds one event it
cannot explain (a pre-existing junk `tmp` row with no recoverable `Source line:` marker),
so by D-18 it ends by raising `CommandError` — under `--dry-run` too. This is the command
working as designed on a database that still holds an unexplainable event, not a notebook
failure: an operator would resolve it in the admin and re-run. Both cells drive the command
through `call_command()` inside a `try`/`except CommandError as exc:` whose handler prints
the message, so the report is shown as real output and the cells after it still execute.

In [3]:
stdout_buf_dry = io.StringIO()
stderr_buf_dry = io.StringIO()
try:
    call_command('cutover_classical_allocations', '--dry-run', stdout=stdout_buf_dry, stderr=stderr_buf_dry)
    dry_run_error = None
except CommandError as exc:
    dry_run_error = exc

print('stdout:', stdout_buf_dry.getvalue())
print('stderr:', stderr_buf_dry.getvalue())
if dry_run_error is not None:
    print(f'CommandError raised (expected, D-18 on this database): {dry_run_error}')
else:
    print('No CommandError raised -- every candidate event was explainable.')

stdout: Done (dry run). candidates: 10, groups: 3, runs created: 3, updated: 0, unchanged: 0, events re-keyed: 9, unexplained: 1
  unexplained (no_source_line): 1 -- no parseable Source line: marker
Done (dry run). total ALLOC:-keyed calendar events now in the database: 48

stderr: pk=334 ('tmp'): no parseable Source line: marker

CommandError raised (expected, D-18 on this database): 1 event(s) could not be explained and were left untouched (no_source_line=1). Resolve the listed events in the admin (see the stderr lines above for each pk, title and reason), then re-run this command -- it is safe to repeat: a repeat pass converts nothing it has not already explained, and updates an existing CampaignRun only when that run's stored Source line: matches the line being converted, reporting anything else instead (NF-19/CR-01, 35-REVIEW.md).


In [4]:
import re

stdout_buf_real = io.StringIO()
stderr_buf_real = io.StringIO()
try:
    call_command('cutover_classical_allocations', stdout=stdout_buf_real, stderr=stderr_buf_real)
    real_run_error = None
except CommandError as exc:
    real_run_error = exc

print('stdout:', stdout_buf_real.getvalue())
print('stderr:', stderr_buf_real.getvalue())
if real_run_error is not None:
    print(f'CommandError raised (expected, D-18 on this database): {real_run_error}')
else:
    print('No CommandError raised -- every candidate event was explainable.')

unexplained_pks = {int(pk) for pk in re.findall(r'pk=(\d+)', stderr_buf_real.getvalue())}
print(f'Unexplained event pk(s) reported by the real cutover: {sorted(unexplained_pks)}')

stdout: Done. candidates: 10, groups: 3, runs created: 3, updated: 0, unchanged: 0, events re-keyed: 9, unexplained: 1
  unexplained (no_source_line): 1 -- no parseable Source line: marker
Done. total ALLOC:-keyed calendar events now in the database: 57

stderr: pk=334 ('tmp'): no parseable Source line: marker

CommandError raised (expected, D-18 on this database): 1 event(s) could not be explained and were left untouched (no_source_line=1). Resolve the listed events in the admin (see the stderr lines above for each pk, title and reason), then re-run this command -- it is safe to repeat: a repeat pass converts nothing it has not already explained, and updates an existing CampaignRun only when that run's stored Source line: matches the line being converted, reporting anything else instead (NF-19/CR-01, 35-REVIEW.md).
Unexplained event pk(s) reported by the real cutover: [334]


### A duplicate run-identity key across two groups (NF-14, 35-REVIEW.md)

`_source_identifier()` deliberately ignores the schedule line's status word, so two
`Source line:` groups that differ only in status (e.g. an `allocation` line and a
`cancelled` line for the same telescope, instrument, window and proposal) resolve to the
SAME run identity key. The cutover now rejects the second group outright -- under its own
`duplicate_identity` reason -- rather than silently merging it into the first group's
`CampaignRun`. This throwaway fixture is created and fully cleaned up inside this one
cell, so it does not affect the before/after counts the rest of this section computes.

In [5]:
from datetime import date, datetime
from datetime import timezone as dup_dt_timezone

from tom_targets.models import TargetList

demo_campaign = TargetList.objects.create(name='NF-14 duplicate-identity demo (throwaway)')
_demo_nights = [date(date.today().year, 7, 9), date(date.today().year, 7, 10), date(date.today().year, 7, 11)]


def _make_demo_event(source_line, status, night):
    start_time = datetime(night.year, night.month, night.day, 23, 0, tzinfo=dup_dt_timezone.utc)
    end_time = datetime(night.year, night.month, night.day + 1, 9, 0, tzinfo=dup_dt_timezone.utc)
    description = (
        f'Dark window (-15 deg, UTC): {start_time.isoformat()} to {end_time.isoformat()}\n'
        f'Status: {status}\nSource line: {source_line}'
    )
    return CalendarEvent.objects.create(
        title='NTT EFOSC2',
        url='',
        description=description,
        telescope='NTT',
        instrument='EFOSC2',
        start_time=start_time,
        end_time=end_time,
        target_list=demo_campaign,
    )


_allocation_line = 'NTT EFOSC2 allocation 9-12 July [NF-14-demo]'
_cancelled_line = 'NTT EFOSC2 cancelled 9-12 July [NF-14-demo]'
demo_events = [_make_demo_event(_allocation_line, 'allocation', night) for night in _demo_nights]
demo_events += [_make_demo_event(_cancelled_line, 'cancelled', night) for night in _demo_nights]
demo_pks = {event.pk for event in demo_events}

stdout_buf_dup_dry = io.StringIO()
stderr_buf_dup_dry = io.StringIO()
try:
    call_command('cutover_classical_allocations', '--dry-run', stdout=stdout_buf_dup_dry, stderr=stderr_buf_dup_dry)
except CommandError as exc:
    print(f'dry run CommandError (expected -- pk=334 plus the duplicate-identity group): {exc}')
print('stdout:', stdout_buf_dup_dry.getvalue())
print('stderr:', stderr_buf_dup_dry.getvalue())

stdout_buf_dup_real = io.StringIO()
stderr_buf_dup_real = io.StringIO()
try:
    call_command('cutover_classical_allocations', stdout=stdout_buf_dup_real, stderr=stderr_buf_dup_real)
except CommandError as exc:
    print(f'real run CommandError (expected -- pk=334 plus the duplicate-identity group): {exc}')
print('stdout:', stdout_buf_dup_real.getvalue())
print('stderr:', stderr_buf_dup_real.getvalue())

dry run CommandError (expected -- pk=334 plus the duplicate-identity group): 4 event(s) could not be explained and were left untouched (duplicate_identity=3, no_source_line=1). Resolve the listed events in the admin (see the stderr lines above for each pk, title and reason), then re-run this command -- it is safe to repeat: a repeat pass converts nothing it has not already explained, and updates an existing CampaignRun only when that run's stored Source line: matches the line being converted, reporting anything else instead (NF-19/CR-01, 35-REVIEW.md).
stdout: Done (dry run). candidates: 7, groups: 2, runs created: 1, updated: 0, unchanged: 0, events re-keyed: 3, unexplained: 4
  unexplained (no_source_line): 1 -- no parseable Source line: marker
  unexplained (duplicate_identity): 3 -- a second Source line resolves to the same run identity key as an earlier group, or a CampaignRun already holds the derived identity key and cannot be proved to have come from this line
Done (dry run). t

### The command's own prescribed re-run is now safe (NF-19, 35-08)

The cutover's `CommandError` tells the operator to resolve the listed events and
re-run the command. Before 35-08's fix, that exact re-run -- invoked a second time
over an unchanged fixture -- silently merged the rejected `duplicate_identity` group
into the winning group's `CampaignRun`. The database-scoped guard now catches it:
a second invocation leaves the winning group's `run_status`, `observation_details`
and `ALLOC:` event titles byte-identical, asserted below rather than only claimed in
prose.

In [6]:
# Demonstrate NF-19 fixed: the cutover's own CommandError prescribes re-running the
# command, and that re-run must not silently merge the rejected duplicate_identity
# group into the winning group's CampaignRun (35-08).
first_group_run = CampaignRun.objects.get(campaign=demo_campaign)
run_status_before = first_group_run.run_status
observation_details_first_line_before = first_group_run.observation_details.splitlines()[0]
alloc_titles_before = sorted(
    CalendarEvent.objects.filter(url__startswith=f'{ALLOC_URL_NAMESPACE}{first_group_run.pk}:').values_list(
        'title', flat=True
    )
)

stdout_buf_dup_second = io.StringIO()
stderr_buf_dup_second = io.StringIO()
try:
    call_command('cutover_classical_allocations', stdout=stdout_buf_dup_second, stderr=stderr_buf_dup_second)
except CommandError as exc:
    print(f'second real-run CommandError (expected -- pk=334 plus the duplicate-identity ' f'group, unchanged): {exc}')
print('stdout:', stdout_buf_dup_second.getvalue())
print('stderr:', stderr_buf_dup_second.getvalue())

first_group_run.refresh_from_db()
run_status_after = first_group_run.run_status
observation_details_first_line_after = first_group_run.observation_details.splitlines()[0]
alloc_titles_after = sorted(
    CalendarEvent.objects.filter(url__startswith=f'{ALLOC_URL_NAMESPACE}{first_group_run.pk}:').values_list(
        'title', flat=True
    )
)

print(f'\nrun_status before second pass: {run_status_before!r}')
print(f'run_status after second pass:  {run_status_after!r}')
print(f'observation_details first line before: {observation_details_first_line_before!r}')
print(f'observation_details first line after:  {observation_details_first_line_after!r}')
print(f'ALLOC: event titles: {alloc_titles_before}')

assert run_status_after == run_status_before, "a second cutover pass changed the winning group's run_status"
assert (
    observation_details_first_line_after == observation_details_first_line_before
), "a second cutover pass changed the winning group's observation_details"
assert (
    alloc_titles_after == alloc_titles_before
), "a second cutover pass changed the winning group's ALLOC: event titles"
print("\nSecond pass confirmed safe: the first group's CampaignRun is unchanged.")

# Cleanup: fully remove this cell's own fixture (the run its winning group created, whose
# pre_delete cascade clears its own ALLOC: events first, plus the rejected group's still-
# blank events and the throwaway campaign) so the sections below see the same before/after
# counts they would without this demonstration.
CampaignRun.objects.filter(campaign=demo_campaign).delete()
CalendarEvent.objects.filter(pk__in=demo_pks).delete()
demo_campaign.delete()
assert not CalendarEvent.objects.filter(pk__in=demo_pks).exists(), 'demo fixture cleanup left a stray event behind'
print('\nDemo fixture fully cleaned up.')

second real-run CommandError (expected -- pk=334 plus the duplicate-identity group, unchanged): 4 event(s) could not be explained and were left untouched (duplicate_identity=3, no_source_line=1). Resolve the listed events in the admin (see the stderr lines above for each pk, title and reason), then re-run this command -- it is safe to repeat: a repeat pass converts nothing it has not already explained, and updates an existing CampaignRun only when that run's stored Source line: matches the line being converted, reporting anything else instead (NF-19/CR-01, 35-REVIEW.md).
stdout: Done. candidates: 4, groups: 1, runs created: 0, updated: 0, unchanged: 0, events re-keyed: 0, unexplained: 4
  unexplained (no_source_line): 1 -- no parseable Source line: marker
  unexplained (duplicate_identity): 3 -- a second Source line resolves to the same run identity key as an earlier group, or a CampaignRun already holds the derived identity key and cannot be proved to have come from this line
Done. to


Demo fixture fully cleaned up.


The fourth step of the cutover sequence: one `reconcile_campaign_runs` sweep takes over
every remaining `RUN:{pk}:{date}` night still reachable by its own run's per-night dispatch
(`rekeyed`), and deletes the rest as one-time churn for a run that now dispatches to a
whole-window container (`legacy_deleted`).

In [7]:
stdout_buf_sweep = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_sweep, stderr=io.StringIO())
print(stdout_buf_sweep.getvalue())
assert 'legacy_deleted' in stdout_buf_sweep.getvalue(), 'expected the sweep summary to report legacy_deleted'

Done. runs: 48, created: 0, updated: 0, unchanged: 73, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, detached: 0, detach_declined: 0, remint_declined: 0, retired: 0, rekeyed: 0, legacy_deleted: 0



In [8]:
after_counts = key_family_counts()
after_campaign_run_count = CampaignRun.objects.count()
after_facility_snapshot = facility_url_snapshot()

print('=== AFTER the cutover + one reconciler sweep ===')
print(f'{"Metric":28}  {"Before":>8}  {"After":>8}')
print('-' * 48)
for key in before_counts:
    print(f'{key:28}  {before_counts[key]:>8}  {after_counts[key]:>8}')
print(f'{"CampaignRun rows":28}  {before_campaign_run_count:>8}  {after_campaign_run_count:>8}')

=== AFTER the cutover + one reconciler sweep ===
Metric                          Before     After
------------------------------------------------
total                              233       233
run_dated (RUN:{pk}:{date})          0         0
run_bare (RUN:{pk})                 16        16
blank_url                           10         1
alloc (ALLOC:)                      48        57
facility_url                       159       159
CampaignRun rows                    45        48


Each of ROADMAP Success Criterion 5's four end-state properties is asserted in an
executed cell below, not just claimed in prose — a future re-execution fails loudly if any
of them regress.

In [9]:
# 1. Zero date-bearing RUN:{pk}:{date} reconciler nights remain.
assert (
    after_counts['run_dated (RUN:{pk}:{date})'] == 0
), 'expected zero date-bearing RUN:{pk}:{date} nights after the cutover + sweep'

# 2. The only blank-url rows remaining are the ones the cutover itself reported as unexplained.
remaining_blank_url_pks = set(CalendarEvent.objects.filter(url='').values_list('pk', flat=True))
print(f'Remaining blank-url event pk(s): {sorted(remaining_blank_url_pks)}')
assert remaining_blank_url_pks == unexplained_pks, (
    'expected every remaining blank-url event to be one the cutover reported as unexplained, '
    f'got remaining={sorted(remaining_blank_url_pks)} vs. reported={sorted(unexplained_pks)}'
)

# 3. The bare RUN:{pk} containers are unchanged in count.
assert (
    after_counts['run_bare (RUN:{pk})'] == before_counts['run_bare (RUN:{pk})']
), 'expected the bare RUN:{pk} container count to be unchanged'

# 4. Every facility-url-keyed observation event is byte-identical before and after.
facility_diff = {
    pk: (before_facility_snapshot.get(pk), after_facility_snapshot.get(pk))
    for pk in set(before_facility_snapshot) | set(after_facility_snapshot)
    if before_facility_snapshot.get(pk) != after_facility_snapshot.get(pk)
}
print(f'Facility-url event differences: {len(facility_diff)}')
assert not facility_diff, f'expected every facility-url event to be byte-identical, found differences: {facility_diff}'

print()
print('All four end-state properties hold: zero date-bearing RUN: nights, only the reported')
print('blank-url row(s) remain unexplained, the bare containers are unchanged in count, and')
print('every facility-url-keyed observation event is byte-identical.')

Remaining blank-url event pk(s): [334]
Facility-url event differences: 0

All four end-state properties hold: zero date-bearing RUN: nights, only the reported
blank-url row(s) remain unexplained, the bare containers are unchanged in count, and
every facility-url-keyed observation event is byte-identical.


## Seed Observatory records and the campaign TargetList

The section above proved the classical cutover; the rest of this notebook demonstrates
`reconcile_run()`'s current dispatch branches on top of the now-cut-over database, using
fresh synthetic fixtures that do not overlap with any real production run.

The reconciler needs one resolvable ground `Observatory` (with a real IANA `timezone`, so
`sun_event()` can compute dip-corrected sunset/sunrise) and one satellite `Observatory`
(`observations_type=SATELLITE_OBSTYPE`, no fixed horizon -- the container branch skips the
per-night sun math for these entirely). `update_or_create` makes this cell idempotent --
safe to re-run against any dev DB.

The campaign container is a `tom_targets.models.TargetList`, found-or-created by name.

In [10]:
from tom_targets.models import TargetList

from solsys_code.solsys_code_observatory.models import Observatory

ground_site, _ = Observatory.objects.update_or_create(
    obscode='X29',
    defaults=dict(
        name='Reconciler Demo Ground Site',
        short_name='RDGS',
        lat=-29.2567,
        lon=-70.7300,
        altitude=2347,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Ground site:    obscode={ground_site.obscode!r}  timezone={ground_site.timezone!r}')

satellite_site, _ = Observatory.objects.update_or_create(
    obscode='X30',
    defaults=dict(
        name='Reconciler Demo Space Telescope',
        short_name='RDST',
        observations_type=Observatory.SATELLITE_OBSTYPE,
    ),
)
print(f'Satellite site: obscode={satellite_site.obscode!r}  observations_type=SATELLITE_OBSTYPE')

campaign, campaign_created = TargetList.objects.get_or_create(name='Reconciler Demo Campaign')
print(f'\nCampaign: {campaign.name!r} (pk={campaign.pk}) {"created" if campaign_created else "found"}')

Ground site:    obscode='X29'  timezone='America/Santiago'
Satellite site: obscode='X30'  observations_type=SATELLITE_OBSTYPE

Campaign: 'Reconciler Demo Campaign' (pk=14) created


## Seed four CampaignRun rows — the four dispatch branches

`reconcile_run()` dispatches on the run's own state, in this order (D-09/D-10, Phase 35): a
non-blank `telescope_class` (class-wide/space) wins first, then a satellite `site`, then a
queue `source` (`lco_queue`/`soar_queue`/`gemini_queue`/`eso_queue`) regardless of site, and
only then the default per-night allocation branch — owned entirely by the peer
`allocation_projector` module — which every other approved, windowed run with a resolved
ground site takes (`web`/`csv_import`/`classical_file`/`legacy`). The four rows below
exercise all four branches:

1. **Class-wide** — `telescope_class` set, no site at all. Projects a single bare
   whole-window `RUN:{pk}` container event.
2. **Satellite** — a `site` with `observations_type=SATELLITE_OBSTYPE` (no fixed horizon).
   Projects a single bare whole-window `RUN:{pk}` container event.
3. **Queue** — `source` set to `CampaignRun.Source.LCO_QUEUE`, with the SAME kind of
   resolved ground site as the classical run below. **As of Phase 35 (D-10), a queue source
   always dispatches to the whole-window container regardless of its site** — a queue
   window is a request, not a set of owned nights, so it is never fanned out into per-night
   events. This is a behaviour change from before Phase 35: the same fixture used to get
   one event per night here.
4. **Campaign-less classical** — a non-queue `source` (`classical_file`), a resolved ground
   site, a 3-night window, and **no campaign** — the common case for a schedule-file run
   (`load_telescope_runs` never sets a campaign unless `--campaign` is given). Dispatches to
   the peer `allocation_projector` module, which projects one `ALLOC:{run_pk}:{night}`
   `CalendarEvent` per observing night.

**Why `source` is still set explicitly on every row, even though it decides only ONE of the
four branches here:** it is the run's provenance record (where it came from), and
staff-facing views and reports read it regardless of which branch it selects.
**What happens if a run's `telescope_class`, `site` or `source` changes after it was
already reconciled once under its old classification:** `reconcile_run()` re-derives which
branch a run belongs to from its *current* state every time it runs, so a later reconcile
automatically detaches (the bare-container family) or deletes (the now-retired date-bearing
family, Phase 35 Task 1) the old family's events rather than leaving them on the calendar
looking like a live commitment forever. See `campaign_reconciler._detach_stale_family_events()`
and "Can I correct a run's source?" in `docs/runbooks/telescope_runs_calendar.rst` for the
full explanation.

In [11]:
from datetime import date

from solsys_code.models import CampaignRun

classical_run, _ = CampaignRun.objects.update_or_create(
    campaign=None,
    telescope_instrument='RDGS/EFOSC2',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 3),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='Classical multi-night photometric monitoring (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
    ),
)
print(
    f'Classical run:  pk={classical_run.pk}  source={classical_run.source!r}  '
    f'campaign={classical_run.campaign!r}  window={classical_run.window_start}..{classical_run.window_end}'
)

queue_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDGS 1m0-SciCam-Sinistro',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 3),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='LCO queue allocation, resolved ground site (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LCO_QUEUE,
    ),
)
print(
    f'Queue run:      pk={queue_run.pk}  source={queue_run.source!r}  '
    f'window={queue_run.window_start}..{queue_run.window_end}  '
    f'(resolved site -- but D-10 sends a queue source straight to the whole-window container)'
)

satellite_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDST Space Telescope',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 5),
    defaults=dict(
        site=satellite_site,
        site_raw='X30',
        observation_details='Satellite allocation, no fixed horizon (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LEGACY,
    ),
)
print(
    f'Satellite run:  pk={satellite_run.pk}  site={satellite_run.site!r}  '
    f'window={satellite_run.window_start}..{satellite_run.window_end}'
)

class_wide_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='LCO 1m0 Network (demo)',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 30),
    defaults=dict(
        site=None,
        telescope_class=CampaignRun.TelescopeClass.ONE_M0,
        observation_details='Class-wide 1m0 network allocation (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LEGACY,
    ),
)
print(
    f'Class-wide run: pk={class_wide_run.pk}  telescope_class={class_wide_run.telescope_class!r}  '
    f'site={class_wide_run.site!r}  window={class_wide_run.window_start}..{class_wide_run.window_end}'
)

Classical run:  pk=73  source=CampaignRun.Source.CLASSICAL_FILE  campaign=None  window=2026-09-01..2026-09-03
Queue run:      pk=74  source=CampaignRun.Source.LCO_QUEUE  window=2026-09-01..2026-09-03  (resolved site -- but D-10 sends a queue source straight to the whole-window container)
Satellite run:  pk=75  site=<Observatory: X30: Reconciler Demo Space Telescope>  window=2026-09-01..2026-09-05
Class-wide run: pk=76  telescope_class=CampaignRun.TelescopeClass.ONE_M0  site=None  window=2026-09-01..2026-09-30


## Dry run first

`--dry-run` reports the `would_create`/`would_update`/`would_leave_unchanged` counters
without writing a single `CalendarEvent` row — always run this before a real sweep.

In [12]:
events_before_dry_run = CalendarEvent.objects.count()

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()
call_command('reconcile_campaign_runs', '--dry-run', stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr:', stderr_buf.getvalue())

events_after_dry_run = CalendarEvent.objects.count()
print(f'CalendarEvent.objects.count() before dry run: {events_before_dry_run}')
print(f'CalendarEvent.objects.count() after dry run:  {events_after_dry_run}')
assert events_after_dry_run == events_before_dry_run, 'A --dry-run sweep must never write a CalendarEvent row'

stdout: Done (dry run). runs: 52, would_create: 6, would_update: 0, would_leave_unchanged: 73, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, would_detach: 0, detach_declined: 0, remint_declined: 0, would_retire: 0, would_rekey: 0, would_delete_legacy: 0

stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)

CalendarEvent.objects.count() before dry run: 233
CalendarEvent.objects.count() after dry run:  233


## The real sweep

Running the same sweep without `--dry-run` now writes the calendar events. The printed
loop below inspects only the four runs seeded above: `owned_events()` (the reconciler's
own `RUN:` namespace) for the class-wide, satellite and queue runs — all three now
dispatch to the whole-window container branch (D-10) — and `allocation_events()` (the peer
`allocation_projector` module's `ALLOC:` namespace) for the campaign-less classical run,
which dispatches to the per-night allocation branch instead.

Submitters write a run's Telescope / Instrument as free text using `/` or `+` (for example
`'RDGS/EFOSC2'`). Both branches split that text on the first delimiter into the calendar
event's two separate `telescope` and `instrument` fields — exactly what the event-detail
pop-up renders as its "Telescope" and "Instrument" boxes. A value with no delimiter at all,
like the satellite and class-wide runs' `telescope_instrument` below, goes wholly into
`telescope` with `instrument` left blank.

In [13]:
from solsys_code.allocation_projector import allocation_events
from solsys_code.campaign_reconciler import owned_events, reconcile_run

stdout_buf_sweep2 = io.StringIO()
stderr_buf_sweep2 = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_sweep2, stderr=stderr_buf_sweep2)

print('stdout:', stdout_buf_sweep2.getvalue())
if stderr_buf_sweep2.getvalue():
    print('stderr:', stderr_buf_sweep2.getvalue())

for label, run, events_fn in [
    ('Class-wide', class_wide_run, owned_events),
    ('Satellite', satellite_run, owned_events),
    ('Queue', queue_run, owned_events),
    ('Classical (campaign-less)', classical_run, allocation_events),
]:
    print(f'--- {label} run (pk={run.pk}) ---')
    for ev in events_fn(run).order_by('start_time'):
        print(f'  url={ev.url!r}')
        print(f'    title={ev.title!r}')
        print(f'    telescope={ev.telescope!r}  instrument={ev.instrument!r}')
        print(f'    start={ev.start_time.isoformat()}  end={ev.end_time.isoformat()}')
    print()

stdout: Done. runs: 52, created: 6, updated: 0, unchanged: 73, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, detached: 0, detach_declined: 0, remint_declined: 0, retired: 0, rekeyed: 0, legacy_deleted: 0

stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)

--- Class-wide run (pk=76) ---
  url='RUN:76'
    title='LCO 1m0 Network (demo) (window 2026-09-01..2026-09-30)'
    telescope='LCO 1m0 Network (demo)'  instrument=''
    start=2026-09-01T00:00:00+00:00  end=2026-09-30T23:59:00+00:00

--- Satellite run (pk=75) ---
  url='RUN:75'
    title='RDST Space Telescope (window 2026-09-01..2026-09-05)'
    telescope='RDST Space Telescope'  instrument=''
    start=2026-09-01T00:00:00+00:00  end=2026-09-05T23:59:00+00:00

--- Queue run (

## Idempotency: a second sweep changes nothing

Running `reconcile_campaign_runs` again against unchanged run state must report
`created: 0, updated: 0` — demonstrated below rather than asserted, per RECON-01.

In [14]:
stdout_buf_second = io.StringIO()
stderr_buf_second = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_second, stderr=stderr_buf_second)

print('Second sweep stdout:', stdout_buf_second.getvalue())
if stderr_buf_second.getvalue():
    print('Second sweep stderr:', stderr_buf_second.getvalue())

assert 'created: 0' in stdout_buf_second.getvalue()
assert 'updated: 0' in stdout_buf_second.getvalue()

Second sweep stdout: Done. runs: 52, created: 0, updated: 0, unchanged: 79, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, detached: 0, detach_declined: 0, remint_declined: 0, retired: 0, rekeyed: 0, legacy_deleted: 0

Second sweep stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)



## Proving namespace isolation: a real sweep touches nothing outside RUN: and ALLOC:

The four runs seeded above already have their `CalendarEvent` rows from the real sweep two
cells up, and the second sweep above already proved that re-running `reconcile_campaign_runs`
against unchanged run state writes nothing. The cells below go one step further, against the
WHOLE scratch copy of the developer database (already cut over above), not just this
notebook's own four demo runs: **every** `CalendarEvent` whose url falls outside BOTH the
reconciler's own `RUN:` namespace AND the allocation projector's `ALLOC:` namespace —
observation events, hand-entered conference/proposal-deadline entries, everything — is
snapshotted, previewed, then swept for real, and the before/after diff over that whole set
must be empty. This extends D-04's real-database proof (Phase 33) to also cover the
allocation projector's own namespace, since Phase 35 gave the sweep a second writer.

Why the preview runs first: after Phase 35 the sweep has TWO writers — the reconciler's
own container branch and the peer allocation projector's per-night branch — so before
trusting a full sweep against this developer's real data, the sweep is previewed in Python
first: every url either writer could touch is computed from their own key builders and
asserted to stay inside `RUN:`/`ALLOC:`, and every pre-existing event outside both
namespaces that is already attributed to a run is asserted to be outside that touchable
set. Only once that preview holds does the real sweep below become a confirmation of an
already-checked claim, rather than the experiment itself.

In [15]:
from datetime import timedelta

from django.core.exceptions import ObjectDoesNotExist

from solsys_code.allocation_projector import allocation_night_url
from solsys_code.campaign_reconciler import run_container_url
from solsys_code.models import CalendarEventMeta
from solsys_code.solsys_code_observatory.models import Observatory


def snapshot_non_sweep_events():
    """(pk -> url, title, description, start_time, end_time, attributed run_id) for every
    CalendarEvent whose url is NOT inside the RUN: or ALLOC: namespaces."""
    snapshot = {}
    qs = (
        CalendarEvent.objects.exclude(url__startswith=RUN_URL_NAMESPACE)
        .exclude(url__startswith=ALLOC_URL_NAMESPACE)
        .select_related('telescope_label_meta')
    )
    for event in qs:
        try:
            run_id = event.telescope_label_meta.run_id
        except ObjectDoesNotExist:
            run_id = None
        snapshot[event.pk] = (event.url, event.title, event.description, event.start_time, event.end_time, run_id)
    return snapshot


# --- Step 1: snapshot every non-RUN:/non-ALLOC:-namespaced CalendarEvent, BEFORE any write below ---
pre_sweep_snapshot = snapshot_non_sweep_events()
print(f'Step 1 -- snapshotted {len(pre_sweep_snapshot)} non-RUN:/non-ALLOC:-namespaced CalendarEvent row(s).')

# --- Step 2: PRE-SWEEP DRY-RUN INSPECTION -- preview the sweep in Python before it runs ---
print()
print('=== PRE-SWEEP DRY-RUN INSPECTION ===')
print(
    f"{'run_pk':>7} {'created':>7} {'updated':>7} {'unchanged':>9} {'blocked':>7} "
    f"{'skipped_nights':>14}  skipped_reason"
)

touchable_urls: set[str] = set()

for run in CampaignRun.objects.all().select_related('site', 'campaign').order_by('pk'):
    try:
        result = reconcile_run(run, dry_run=True)
    except Exception as exc:  # noqa: BLE001 -- mirrors reconcile_campaign_runs.handle()'s own per-run catch
        print(f'{run.pk:>7}  reconcile_run(dry_run=True) raised {exc!r} -- skipping')
        continue

    print(
        f'{run.pk:>7} {result.created:>7} {result.updated:>7} {result.unchanged:>9} '
        f'{result.blocked:>7} {result.skipped_nights:>14}  {result.skipped_reason or ""}'
    )

    if result.skipped_reason is not None:
        # Stage-0 guard fired (_skip_reason()) -- this run projects nothing, so it adds no
        # url to the touchable set.
        continue

    if run.telescope_class or (run.site is not None and run.site.observations_type == Observatory.SATELLITE_OBSTYPE):
        touchable_urls.add(run_container_url(run))
    elif run.source in {
        CampaignRun.Source.LCO_QUEUE,
        CampaignRun.Source.SOAR_QUEUE,
        CampaignRun.Source.GEMINI_QUEUE,
        CampaignRun.Source.ESO_QUEUE,
    }:
        touchable_urls.add(run_container_url(run))
    elif run.window_start is not None and run.window_end is not None:
        n_nights = (run.window_end - run.window_start).days + 1
        touchable_urls |= {allocation_night_url(run, run.window_start + timedelta(days=i)) for i in range(n_nights)}

    # owned_events()/allocation_events() are namespace-identity, so this can only ever add
    # RUN:- or ALLOC:-prefixed urls -- included so the detach/retire step's reach is covered.
    touchable_urls |= {e.url for e in owned_events(run)}
    touchable_urls |= {e.url for e in allocation_events(run)}

print()
non_namespace_in_touchable = {
    u for u in touchable_urls if not (u.startswith(RUN_URL_NAMESPACE) or u.startswith(ALLOC_URL_NAMESPACE))
}
print(
    f'Touchable url set size: {len(touchable_urls)}; every member starts with RUN: or ALLOC:: '
    f'{not non_namespace_in_touchable}'
)
assert not non_namespace_in_touchable, f'Touchable set contains urls outside RUN:/ALLOC:: {non_namespace_in_touchable}'

# The direct check on any pre-existing attributed non-RUN:/non-ALLOC: row: every
# CalendarEventMeta whose run is set and whose event's url is OUTSIDE both namespaces must
# never be a url the sweep can touch.
foreign_attributed = (
    CalendarEventMeta.objects.filter(run__isnull=False)
    .exclude(event__url__startswith=RUN_URL_NAMESPACE)
    .exclude(event__url__startswith=ALLOC_URL_NAMESPACE)
    .select_related('event', 'run')
)
print()
print('CalendarEventMeta rows attributed to a run whose event url is OUTSIDE RUN:/ALLOC::')
foreign_urls = set()
for meta in foreign_attributed:
    print(f'  event pk={meta.event_id}  url={meta.event.url!r}  attributed to run pk={meta.run_id}')
    foreign_urls.add(meta.event.url)
if not foreign_urls:
    print('  (none)')

overlap = foreign_urls & touchable_urls
print(f'Any of those urls in the touchable set: {bool(overlap)}')
assert not overlap, f'The sweep must never be able to touch an attributed non-RUN:/non-ALLOC: event: {overlap}'

# Cross-check: the management command's own --dry-run output is an AGGREGATE summary only.
stdout_buf_preview = io.StringIO()
call_command('reconcile_campaign_runs', '--dry-run', stdout=stdout_buf_preview)
print()
print('Aggregate summary only (reconcile_campaign_runs --dry-run stdout) -- the per-run detail')
print('and the url-namespace assertions above come from reconcile_run()s returned ReconcileResult,')
print('not from parsing this line:')
print(stdout_buf_preview.getvalue())

Step 1 -- snapshotted 160 non-RUN:/non-ALLOC:-namespaced CalendarEvent row(s).

=== PRE-SWEEP DRY-RUN INSPECTION ===
 run_pk created updated unchanged blocked skipped_nights  skipped_reason
      1       0       0        15       0              0  
      2       0       0         1       0              0  
      3       0       0         1       0              0  
      4       0       0         0       0              0  TBD window


      5       0       0         1       0              0  
      6       0       0         1       0              0  


      7       0       0         1       0              0  
      8       0       0         1       0              0  
      9       0       0         1       0              0  
     10       0       0         1       0              0  
     11       0       0         1       0              0  
     12       0       0         1       0              0  
     13       0       0         1       0              0  


     14       0       0         1       0              0  
     15       0       0         1       0              0  


     16       0       0         1       0              0  
     17       0       0         1       0              0  
     18       0       0         1       0              0  
     19       0       0         1       0              0  
     20       0       0         1       0              0  
     21       0       0         1       0              0  
     22       0       0         1       0              0  


...

  event pk=334  url=''  attributed to run pk=68
  event pk=350  url='https://observe.lco.global/requests/4229878'  attributed to run pk=1
Any of those urls in the touchable set: False


Run pk=4: skipped (TBD window)


Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)


Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)



Aggregate summary only (reconcile_campaign_runs --dry-run stdout) -- the per-run detail
and the url-namespace assertions above come from reconcile_run()s returned ReconcileResult,
not from parsing this line:
Done (dry run). runs: 52, would_create: 0, would_update: 0, would_leave_unchanged: 79, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, would_detach: 0, detach_declined: 0, remint_declined: 0, would_retire: 0, would_rekey: 0, would_delete_legacy: 0



In [16]:
# --- Step 3: run the real full sweep, then compute the before/after diff ---
stdout_buf_realsweep2 = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_realsweep2)
print('Real sweep stdout:', stdout_buf_realsweep2.getvalue())

post_sweep_snapshot = snapshot_non_sweep_events()

print()
print('=== POST-SWEEP DIFF ===')
compared_pks = set(pre_sweep_snapshot) | set(post_sweep_snapshot)
diff = {
    pk: (pre_sweep_snapshot.get(pk), post_sweep_snapshot.get(pk))
    for pk in compared_pks
    if pre_sweep_snapshot.get(pk) != post_sweep_snapshot.get(pk)
}
print(
    f'Compared {len(compared_pks)} non-RUN:/non-ALLOC:-namespaced CalendarEvent row(s) '
    'before vs. after the real sweep.'
)
print(f'Differences found: {len(diff)}')
for pk, (before, after) in diff.items():
    print(f'  pk={pk}: before={before!r}  after={after!r}')
assert not diff, f'The real sweep must never change a non-RUN:/non-ALLOC:-namespaced event: {diff}'
print('POST-SWEEP DIFF is empty -- the sweep changed nothing outside RUN:/ALLOC:.')

Run pk=4: skipped (TBD window)


Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)


Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)


Real sweep stdout: Done. runs: 52, created: 0, updated: 0, unchanged: 79, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, detached: 0, detach_declined: 0, remint_declined: 0, retired: 0, rekeyed: 0, legacy_deleted: 0


=== POST-SWEEP DIFF ===
Compared 160 non-RUN:/non-ALLOC:-namespaced CalendarEvent row(s) before vs. after the real sweep.
Differences found: 0
POST-SWEEP DIFF is empty -- the sweep changed nothing outside RUN:/ALLOC:.


## Six counters on ReconcileResult

`reconcile_campaign_runs` and `ReconcileResult` report six numbers alongside
`created`/`updated`/`unchanged`/`blocked`:

- **`skipped_nights`** — the classical per-night skip-the-night rule this counter measured
  (D-01, ANNOT-01, Phase 33) is now dead code: `_attributed_nights()` is defined but never
  called, superseded by the allocation projector's own D-05/D-07 handoff (demonstrated
  below), which works by deleting and re-creating an `ALLOC:` night rather than skipping the
  reconciler's own write. `skipped_nights` therefore reports `0` for every run, permanently
  — kept in the API for backward compatibility, not because anything can still trigger it.
- **`detached`** — unchanged in meaning from Phase 33: companion rows the sweep released
  back into Phase 28's attribution queue because a run was re-classified into a different
  dispatch branch.
- **`detach_declined`** now counts ONLY declines caused by a human confirmation
  (`confirmed_by`): a companion row the sweep did not release for the reason above because a
  staff member had already confirmed it, and — new this round (plan 35-23's CR-05, closing
  35-REVIEW.md CR-05) — an allocation night the sweep did not DELETE when a linked
  observation would otherwise have retired it, because that night's companion row was
  confirmed too. A confirmed night now survives its own retirement, at the cost of a
  duplicate calendar entry (the confirmed allocation night alongside the observation's own
  entry) until the confirmation is cleared.
- **`remint_declined`** is a NEW counter this round (plan 35-23's WR-06 split, closing
  35-REVIEW.md WR-06): a re-mint the sweep declined to PERFORM, because the night carries a
  human confirmation, a real `ObservationRecord`/`ObservationGroup` link, or an unverified
  companion row (`is_verified=False`). It is deliberately separate from `detach_declined` —
  a re-mint decline destroys nothing and releases no attribution, so folding it into
  `detach_declined` told an operator the wrong thing happened (35-REVIEW.md WR-06). Since
  plan 35-23's CR-04 fix, a declined re-mint ALSO falls through to the ordinary plain-update
  path, so the same night reports `remint_declined` alongside `updated` on the same sweep —
  its title/description/target_list still refresh, only its boundary rewrite is declined.
- **`retired`**/**`rekeyed`**/**`legacy_deleted`** — the three original Phase 35 additions.
  `rekeyed` and `legacy_deleted` are demonstrated by the cutover section above; `retired`
  counts a night removed from the calendar for any of five reasons (35-REVIEW.md
  NF-07) — a linked `ObservationRecord` placed or observed its block on that night (the
  case demonstrated below, where unlinking restores the night), a sub-night window field
  changed, a correction to the run's `site` (now three distinct outcomes depending on
  whether the sub-night window is fully set — plan 35-24's WR-05 fix, demonstrated below for
  the escalated in-place-`Observatory`-correction case), the night fell out of the run's
  window entirely (a shrink or re-classification), a leftover night whose companion row was
  deleted or had its `run` cleared (35-REVIEW.md NF-01), or a one-time provenance audit
  resolving a night minted before this release (now repeated once more for the `v2`-format
  token this release's `v3` bump leaves behind, also checking the site's own stored position
  and timezone — plan 35-24's escalated decision, demonstrated below).
- **`is_verified`** (on `CalendarEventMeta`, not a `ReconcileResult` field, but the source of
  the third `remint_declined` cause): unchecking it in the Django admin permanently vetoes an
  automated re-mint of that night's boundaries, but does NOT veto its retirement when a
  linked observation places a block on it (plan 35-24's WR-08 fix) — see the runbook's
  `remint_declined` section for the field's exact shipped label and help text.

See `docs/runbooks/telescope_runs_calendar.rst`'s counter section for the full,
operator-facing version of every claim above, in the same words.


## The observation handoff: a linked, placed record retires a night; unlinking restores it

D-05/D-07 (Phase 35): an allocation night with a `CampaignRunObservation` link whose
record's block has been placed or observed has no allocation event at all — the
observation's own event is that night's entry now. This is the exact property spike 003
measured (link -> 1 retired, re-project -> unchanged, unlink -> 1 created), reused
verbatim for the allocation layer.

`classical_run`'s middle night (2026-09-02) already has its own `ALLOC:{pk}:2026-09-02`
event from this notebook's earlier real sweep. Linking a real `ObservationRecord` (a
`NonSiderealTargetFactory` target, per CLAUDE.md — FOMO is exclusively for Solar System
targets) whose `scheduled_start`/`scheduled_end` fall on that same night, then reconciling
`classical_run` again, retires the night: the `ALLOC:` event is deleted, and
`ReconcileResult.retired` counts it — and stays counted as retired on every further
reconcile while the link remains (D-05's "this night IS retired", not "a delete just
happened"). Deleting the `CampaignRunObservation` link restores the night immediately,
with no explicit reconcile call needed at all: D-11's `post_delete` receiver on
`CampaignRunObservation` re-projects the linked run as part of the delete itself, so the
`ALLOC:` event already exists, keyed and titled exactly as it was before the handoff, by
the time `.delete()` returns.

In [17]:
from datetime import datetime
from datetime import timezone as dt_timezone
from uuid import uuid4

from django.contrib.auth.models import User
from tom_observations.models import ObservationRecord
from tom_targets.tests.factories import NonSiderealTargetFactory

from solsys_code.models import CampaignRunObservation

handoff_night = classical_run.window_start + timedelta(days=1)  # 2026-09-02
handoff_url = allocation_night_url(classical_run, handoff_night)

before_link_event = CalendarEvent.objects.get(url=handoff_url)
before_link_pk = before_link_event.pk
print(f'classical_run pk={classical_run.pk} owns {handoff_url!r} (pk={before_link_pk}) for the demo night.')

target = NonSiderealTargetFactory.create()
owner = User.objects.create(username=f'handoff-demo-owner-{uuid4().hex[:8]}')
scheduled_start = datetime(2026, 9, 2, 23, 0, tzinfo=dt_timezone.utc)
scheduled_end = scheduled_start + timedelta(hours=1)
record = ObservationRecord.objects.create(
    target=target,
    user=owner,
    facility='LCO',
    observation_id=f'handoff-demo-{uuid4().hex[:8]}',
    status='COMPLETED',
    scheduled_start=scheduled_start,
    scheduled_end=scheduled_end,
    parameters={'proposal': 'DEMO'},
)
link = CampaignRunObservation.objects.create(run=classical_run, observation_record=record)
print(
    f'Linked ObservationRecord pk={record.pk} (scheduled {scheduled_start.isoformat()} .. '
    f'{scheduled_end.isoformat()}) to classical_run pk={classical_run.pk} via '
    f'CampaignRunObservation pk={link.pk}'
)

retire_result = reconcile_run(classical_run)
print()
print(f'Reconcile result after linking: {retire_result}')
assert retire_result.retired == 1, f'expected exactly 1 retired night, got {retire_result.retired}'
assert not CalendarEvent.objects.filter(url=handoff_url).exists(), 'the retired night must have no allocation event'
print(f'ALLOC: event for {handoff_url!r} exists: {CalendarEvent.objects.filter(url=handoff_url).exists()}')

# Re-projecting again with the link still in place still reports the night as retired --
# retired_nights() reports every currently-retired night, not just newly-retired ones (the
# plan's own Task 2 Test 1: retired means "this night is retired", not "a delete just
# happened") -- so this is idempotent convergence, not a second retirement.
unchanged_result = reconcile_run(classical_run)
print(f'Reconcile result on a further call (link unchanged): {unchanged_result}')
assert unchanged_result.retired == 1, 'the night must still report retired while the link is still in place'
assert not CalendarEvent.objects.filter(url=handoff_url).exists()

link_pk = link.pk
link.delete()
print()
print(f'Deleted CampaignRunObservation pk={link_pk} -- the link is gone.')

# D-11's post_delete receiver on CampaignRunObservation re-projects the linked run
# immediately, so the night is already restored by the time delete() returns -- no
# operator command, no explicit reconcile_run() call needed for the common case.
restored_event = CalendarEvent.objects.get(url=handoff_url)
print(
    f'ALLOC: event for {handoff_url!r} restored automatically by the delete receiver: '
    f'pk={restored_event.pk}  title={restored_event.title!r}'
)
assert (
    restored_event.title == before_link_event.title
), 'the restored night must carry the same title as before the handoff'

# A further explicit reconcile_run() call is now a no-op -- the night is already converged.
further_result = reconcile_run(classical_run)
print(f'A further explicit reconcile_run() call: {further_result}')
assert (
    further_result.created == 0 and further_result.retired == 0
), 'the night is already restored -- a further reconcile must be a no-op'

Target post save hook: lhQITZgtDTutagMYqJQA created: True


classical_run pk=73 owns 'ALLOC:73:2026-09-02' (pk=364) for the demo night.


Target post save hook: lhQITZgtDTutagMYqJQA created: False


No Profile found for handoff-demo-owner-b517d9aa. Creating Profile.


unprojectable observation_id='handoff-demo-5246b997': InstrumentExtractionError: No recognized configuration_type or exposure signal found in observation_id='handoff-demo-5246b997' parameters


Observation change state hook: lhQITZgtDTutagMYqJQA @ LCO from None to COMPLETED


Linked ObservationRecord pk=223 (scheduled 2026-09-02T23:00:00+00:00 .. 2026-09-03T00:00:00+00:00) to classical_run pk=73 via CampaignRunObservation pk=2



Reconcile result after linking: ReconcileResult(created=0, updated=0, unchanged=2, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=1, rekeyed=0, legacy_deleted=0, skipped_reason=None)
ALLOC: event for 'ALLOC:73:2026-09-02' exists: False
Reconcile result on a further call (link unchanged): ReconcileResult(created=0, updated=0, unchanged=2, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=1, rekeyed=0, legacy_deleted=0, skipped_reason=None)



Deleted CampaignRunObservation pk=2 -- the link is gone.
ALLOC: event for 'ALLOC:73:2026-09-02' restored automatically by the delete receiver: pk=369  title='RDGS EFOSC2'


A further explicit reconcile_run() call: ReconcileResult(created=0, updated=0, unchanged=3, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=0, rekeyed=0, legacy_deleted=0, skipped_reason=None)


## A site correction re-mints an already-projected run's nights (plan 35-21, CR-02)

35-REVIEW.md iteration 8's CR-02: before plan 35-21, the recorded mint-provenance token
carried only the sub-night window pair, so correcting `CampaignRun.site` on an
already-projected run compared equal to itself and reported `unchanged` forever — the
night kept its stale boundary from the OLD site's `sun_event()` result, silently. The
widened token now also carries `run.site_id`, so a site correction is detected and the
affected nights re-mint to the new site's real sunset/sunrise, exactly as the reviewer's
probe 1 (La Silla → Siding Spring) demonstrated.

`classical_run`'s first night (its `window_start`, untouched by the observation-handoff
demo above) already has an `ALLOC:` event from the real sweep earlier in this notebook,
minted against `ground_site`'s (`X29`, `America/Santiago`) `sun_event()` values. Below,
`classical_run.site` is corrected to a second real ground site (`E10`, Siding Spring —
the same obscode 35-21's own reviewer probe and `TestSiteChangeRemints` use), and
`reconcile_run()` is called again.

In [18]:
from solsys_code.allocation_projector import night_bounds
from solsys_code.telescope_runs import sun_event

corrected_site, _ = Observatory.objects.update_or_create(
    obscode='E10',
    defaults=dict(
        name='Siding Spring Observatory',
        short_name='FTS',
        lat=-31.2734,
        lon=149.0612,
        altitude=1149,
        timezone='Australia/Sydney',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Corrected ground site: obscode={corrected_site.obscode!r}  timezone={corrected_site.timezone!r}')

site_correction_night = classical_run.window_start  # 2026-09-01, untouched by the handoff demo above
site_correction_url = allocation_night_url(classical_run, site_correction_night)

event_before_correction = CalendarEvent.objects.get(url=site_correction_url)
pk_before_correction = event_before_correction.pk
start_before_correction = event_before_correction.start_time
end_before_correction = event_before_correction.end_time
print(
    f'Before site correction: pk={pk_before_correction}  '
    f'start={start_before_correction.isoformat()}  end={end_before_correction.isoformat()}'
)

classical_run.site = corrected_site
classical_run.site_raw = 'E10'
classical_run.save(update_fields=['site', 'site_raw'])

site_correction_result = reconcile_run(classical_run)
print(f'Reconcile result after the site correction: {site_correction_result}')

event_after_correction = CalendarEvent.objects.get(url=site_correction_url)
expected_sunset, expected_sunrise = sun_event(corrected_site, site_correction_night, kind='sun')
expected_start, expected_end = night_bounds(classical_run, site_correction_night, expected_sunset, expected_sunrise)
print(
    f'After site correction:  pk={event_after_correction.pk}  '
    f'start={event_after_correction.start_time.isoformat()}  end={event_after_correction.end_time.isoformat()}'
)
print(
    f"Corrected site's own sun_event() boundary:      "
    f'start={expected_start.isoformat()}  end={expected_end.isoformat()}'
)

assert (
    event_after_correction.pk != pk_before_correction
), 'a site correction must re-mint (a new primary key), not report unchanged forever'
assert event_after_correction.start_time != start_before_correction, 'the boundary must actually move'
assert event_after_correction.start_time == expected_start
assert event_after_correction.end_time == expected_end
print("Boundaries moved to the corrected site's real sun_event() values, as expected (CR-02 closed).")

Corrected ground site: obscode='E10'  timezone='Australia/Sydney'
Before site correction: pk=363  start=2026-09-01T22:35:09+00:00  end=2026-09-02T10:49:50+00:00


Reconcile result after the site correction: ReconcileResult(created=3, updated=0, unchanged=0, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=3, rekeyed=0, legacy_deleted=0, skipped_reason=None)


After site correction:  pk=370  start=2026-09-01T07:52:18+00:00  end=2026-09-01T20:14:41+00:00
Corrected site's own sun_event() boundary:      start=2026-09-01T07:52:18+00:00  end=2026-09-01T20:14:41+00:00
Boundaries moved to the corrected site's real sun_event() values, as expected (CR-02 closed).


## A declined re-mint declines only the boundary rewrite, and survives its own retirement (plan 35-23, CR-01/CR-04/CR-05)

35-REVIEW.md iteration 8's CR-01: before plan 35-20, the re-mint branch was the only
delete path in `allocation_projector.py` with no human-confirmation guard — a staff
member confirming a night's attribution did not stop an automated sweep from deleting and
re-creating it (with a fresh, unconfirmed companion row) the next time its sub-night window
field changed. `_remint_decline_reason()` now checks before any destructive branch runs,
and a confirmed night's boundary and primary key survive.

Iteration 9's CR-04 (plan 35-23) then closed a narrower gap: the ORIGINAL fix declined the
re-mint by `continue`-ing out of the per-night loop entirely, which also froze the night's
title/description forever — a run later marked `CANCELLED` never showed `[C]` on a
declined night. The decline now refuses only the DESTRUCTIVE half (the delete/create pair
and the boundary rewrite); the night still falls through to the ordinary plain-update path
below it, so its labelling keeps refreshing. The cell below demonstrates both properties
together on the SAME night: the boundary-affecting edit is declined (reported under the
NEW `remint_declined` counter, not `detach_declined` — plan 35-23's WR-06 split, see the
counters section above), while the `CANCELLED` marker still lands on the title, alongside
`updated: 1` on the very same `ReconcileResult`.

A dedicated single-night run is seeded below (rather than reusing `classical_run`, whose
nights the cells above already exercise) with a sub-night window override set from the
start, mirroring `TestDeclinedRemintStillUpdatesLabels`'s own fixture shape in
`test_allocation_projector.py`.


In [19]:
from datetime import time as dt_time

from django.contrib.auth.models import User
from django.utils import timezone as dj_timezone

from solsys_code.models import CampaignRun

declined_run, _ = CampaignRun.objects.update_or_create(
    campaign=None,
    telescope_instrument='RDGS/EFOSC2 (declined re-mint demo)',
    window_start=date(2026, 9, 10),
    window_end=date(2026, 9, 10),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='Single-night run for the declined-re-mint demo',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
        night_start_utc=dt_time(23, 0),
        night_end_utc=dt_time(5, 0),
    ),
)
print(f'Declined-re-mint demo run: pk={declined_run.pk}  window={declined_run.window_start}..{declined_run.window_end}')

decline_night = declined_run.window_start
decline_url = allocation_night_url(declined_run, decline_night)

reconcile_run(declined_run)
declined_event_before = CalendarEvent.objects.get(url=decline_url)
declined_pk_before = declined_event_before.pk
declined_start_before = declined_event_before.start_time
declined_end_before = declined_event_before.end_time
declined_title_before = declined_event_before.title
print(
    f'Minted night:  pk={declined_pk_before}  title={declined_title_before!r}  '
    f'start={declined_start_before.isoformat()}  end={declined_end_before.isoformat()}'
)

staff_user, _ = User.objects.get_or_create(username='reconciler-demo-confirming-staffer')
CalendarEventMeta.objects.filter(event=declined_event_before).update(
    confirmed_by=staff_user, confirmed_at=dj_timezone.now()
)
print(f'Confirmed by: {staff_user.username!r}')

# Change the sub-night window override (would otherwise re-mint the night) AND the run's
# status (which changes the title via the ordinary plain-update path) in the same save --
# CR-04's own fixture shape (TestDeclinedRemintStillUpdatesLabels).
declined_run.night_start_utc = None
declined_run.run_status = CampaignRun.RunStatus.CANCELLED
declined_run.save(update_fields=['night_start_utc', 'run_status'])

declined_result = reconcile_run(declined_run)
print(f'Reconcile result after the boundary-changing AND title-changing edit: {declined_result}')

declined_event_after = CalendarEvent.objects.get(url=decline_url)
declined_meta_after = CalendarEventMeta.objects.get(event=declined_event_after)

assert (
    declined_result.remint_declined == 1
), f'expected exactly 1 declined re-mint, got {declined_result.remint_declined}'
assert declined_result.detach_declined == 0, 'this decline is a re-mint decline, not an attribution/retirement decline'
assert declined_result.updated == 1, 'CR-04: the plain-update path must still run for a declined night'
assert declined_result.retired == 0
assert declined_result.created == 0
assert declined_event_after.pk == declined_pk_before, 'a declined re-mint must keep the same primary key'
assert declined_event_after.start_time == declined_start_before, 'a declined re-mint must keep the same boundary'
assert declined_event_after.end_time == declined_end_before
assert declined_event_after.title != declined_title_before, 'CR-04: the title must still refresh on a declined night'
assert declined_event_after.title.startswith('[C]'), 'the C marker must land on the declined night'
assert declined_meta_after.confirmed_by_id == staff_user.pk
assert declined_meta_after.confirmed_at is not None
print(f'Title after: {declined_event_after.title!r}')
print(
    'Primary key, boundaries and the confirmation all survived the declined re-mint (CR-01 closed); '
    'the title still refreshed on the same sweep (CR-04 closed).'
)

Declined-re-mint demo run: pk=77  window=2026-09-10..2026-09-10


No Profile found for reconciler-demo-confirming-staffer. Creating Profile.


Allocation re-mint declined: event pk=373 night=2026-09-10 is human-confirmed to run pk=77 -- an automated re-mint never clears it.


Minted night:  pk=373  title='RDGS EFOSC2 (declined re-mint demo)'  start=2026-09-10T23:00:00+00:00  end=2026-09-11T05:00:00+00:00
Confirmed by: 'reconciler-demo-confirming-staffer'
Reconcile result after the boundary-changing AND title-changing edit: ReconcileResult(created=0, updated=1, unchanged=0, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=1, retired=0, rekeyed=0, legacy_deleted=0, skipped_reason=None)
Title after: '[C] RDGS EFOSC2 (declined re-mint demo)'
Primary key, boundaries and the confirmation all survived the declined re-mint (CR-01 closed); the title still refreshed on the same sweep (CR-04 closed).


## A confirmed allocation night survives its own retirement (plan 35-23, CR-05), and still gets its labels refreshed (iteration 10, CR-01)

35-REVIEW.md iteration 9's CR-05: before this round, the retirement branch's own
`existing.delete()` was the one delete path in `allocation_projector.py` with NO
human-confirmation guard — a staff member confirming an allocation night's attribution did
not stop the sweep from deleting it the moment a real observation linked to the same night,
taking the companion row's `confirmed_by`/`confirmed_at` and both observation links with it
via `CalendarEventMeta.event`'s cascading delete. Only `confirmed_by` declines a retirement
— deliberately narrower than the re-mint branch's guard above, which also vetoes on
`is_verified=False` or a bare observation link: the re-mint branch destroys a row it intends
to immediately re-create and owes its contents a decision, while the retirement branch
removes a night genuinely superseded by the linked observation's own calendar entry, so
widening its veto the same way would leave a permanent duplicate night beside every
observation-superseded night, not just the confirmed ones.

The consequence an operator actually sees, stated in the runbook's `detach_declined`
section in the same words: the calendar shows BOTH the confirmed allocation night and the
linked observation's own entry, on the same night, until someone clears the confirmation on
that night's companion row and re-runs the sweep. The demo below confirms a fresh
single-night run's allocation night, then links an `ObservationRecord` whose block is
already placed on that same night — the retirement guard fires from the
`CampaignRunObservation` `post_save` receiver alone, with no explicit `reconcile_run()`
call, exactly like `TestRetirePathAllocationEventGuard.test_retiring_a_night_never_deletes_a_human_confirmed_alloc_event`.

35-REVIEW.md iteration 10's CR-01 found that the night this guard keeps alive was then never refreshed again, so a later `Mark cancelled` never reached it; the block below now also proves the surviving night keeps receiving its ordinary title/description/campaign-label refresh, with its primary key, both boundaries and its confirmation stamp preserved, reported as `detach_declined` alongside `updated`.

In [20]:
from datetime import datetime
from datetime import timezone as dt_timezone
from uuid import uuid4

from django.contrib.auth.models import User
from django.utils import timezone as dj_timezone
from tom_observations.models import ObservationRecord
from tom_targets.tests.factories import NonSiderealTargetFactory

from solsys_code.models import CampaignRun, CampaignRunObservation

confirmed_retire_run, _ = CampaignRun.objects.update_or_create(
    campaign=None,
    telescope_instrument='RDGS/EFOSC2 (confirmed-retirement demo)',
    window_start=date(2026, 9, 11),
    window_end=date(2026, 9, 11),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='Single-night run for the confirmed-retirement demo',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
    ),
)
print(f'Confirmed-retirement demo run: pk={confirmed_retire_run.pk}  window={confirmed_retire_run.window_start}')

retire_night = confirmed_retire_run.window_start
retire_url = allocation_night_url(confirmed_retire_run, retire_night)

reconcile_run(confirmed_retire_run)
retire_event_before = CalendarEvent.objects.get(url=retire_url)
retire_pk_before = retire_event_before.pk
print(f'Minted night: pk={retire_pk_before}  url={retire_url!r}')

retire_staff_user, _ = User.objects.get_or_create(username='reconciler-demo-retirement-confirming-staffer')
CalendarEventMeta.objects.filter(event=retire_event_before).update(
    confirmed_by=retire_staff_user, confirmed_at=dj_timezone.now()
)
print(f'Confirmed by: {retire_staff_user.username!r}')

retire_target = NonSiderealTargetFactory.create()
retire_owner = User.objects.create(username=f'retirement-demo-owner-{uuid4().hex[:8]}')
retire_scheduled_start = datetime(2026, 9, 11, 23, 30, tzinfo=dt_timezone.utc)
retire_record = ObservationRecord.objects.create(
    target=retire_target,
    user=retire_owner,
    facility='LCO',
    observation_id=f'retirement-demo-{uuid4().hex[:8]}',
    status='COMPLETED',
    scheduled_start=retire_scheduled_start,
    scheduled_end=retire_scheduled_start + timedelta(hours=1),
    parameters={'proposal': 'DEMO'},
)
retire_link = CampaignRunObservation.objects.create(run=confirmed_retire_run, observation_record=retire_record)
print(
    f'Linked ObservationRecord pk={retire_record.pk} to confirmed_retire_run pk={confirmed_retire_run.pk} '
    f'via CampaignRunObservation pk={retire_link.pk} -- the retirement guard fires here, from the '
    'receiver alone, before any explicit reconcile_run() call.'
)

# The allocation night's own event, companion row and confirmation stamp must all still
# exist -- the guard declined the delete, it did not merely delay it.
assert CalendarEvent.objects.filter(pk=retire_pk_before).exists(), 'the confirmed allocation night must survive'
retire_meta_after = CalendarEventMeta.objects.get(event_id=retire_pk_before)
assert retire_meta_after.confirmed_by_id == retire_staff_user.pk
assert retire_meta_after.confirmed_at is not None
print(
    f'Allocation night pk={retire_pk_before} still exists after the link, with its confirmation intact '
    f'(confirmed_by={retire_meta_after.confirmed_by.username!r}).'
)

# An explicit reconcile_run() call reproduces the same already-applied decision, and its
# counters (unavailable from the receiver itself, which returns nothing) can be captured.
retire_result = reconcile_run(confirmed_retire_run)
print(f'Reconcile result reproducing the same decision: {retire_result}')
assert retire_result.retired == 0, 'a confirmed allocation night must never be counted as retired'
assert (
    retire_result.detach_declined == 1
), 'the declined retirement is counted under detach_declined, not remint_declined'
print(
    "The calendar now shows BOTH the confirmed allocation night AND the linked observation's own "
    "entry on the same night -- the duplicate the runbook's detach_declined section describes, "
    'resolved only by clearing the confirmation and re-running the sweep.'
)
# 35-REVIEW.md iteration 10, CR-01: the surviving night above must also keep receiving its
# ordinary label refresh -- the retirement decline refused only the DELETE, never the
# title/description/target_list update.
from solsys_code.allocation_projector import allocation_night_title

retire_title_before = retire_event_before.title
retire_start_before = retire_event_before.start_time
retire_end_before = retire_event_before.end_time
print(f'Title before the cancellation sweep: {retire_title_before!r}')

confirmed_retire_run.run_status = CampaignRun.RunStatus.CANCELLED
confirmed_retire_run.save(update_fields=['run_status'])

retire_refresh_result = reconcile_run(confirmed_retire_run)
print(f'Reconcile result after marking the run cancelled: {retire_refresh_result}')

retire_event_after = CalendarEvent.objects.get(pk=retire_pk_before)
print(f'Title after the cancellation sweep: {retire_event_after.title!r}')

assert retire_event_after.title == allocation_night_title(
    confirmed_retire_run
), 'the surviving night must carry the current allocation_night_title()'
assert retire_event_after.title.startswith(
    '[C]'
), 'the surviving night must pick up the [C] marker on the sweep that cancels its run'
assert retire_event_after.start_time == retire_start_before, 'the label refresh must never rewrite start_time'
assert retire_event_after.end_time == retire_end_before, 'the label refresh must never rewrite end_time'
retire_meta_refreshed = CalendarEventMeta.objects.get(event_id=retire_pk_before)
assert (
    retire_meta_refreshed.confirmed_by_id == retire_staff_user.pk
), 'the label refresh must never clear the confirmation stamp'
assert retire_refresh_result.detach_declined == 1, 'the declined retirement is still counted under detach_declined'
assert (
    retire_refresh_result.updated == 1
), 'the label refresh is counted under updated on the sweep that changes the labels'
assert retire_refresh_result.retired == 0, 'a confirmed allocation night must never be counted as retired'
print(
    f'Night pk={retire_pk_before} still receives the ordinary label refresh -- its primary key, '
    'both boundaries and its confirmation stamp are preserved, while its title picks up the '
    '[C] prefix.'
)

# A further sweep with nothing else changed converges: the decline (and its warning) repeats,
# but the labels are already refreshed, so this reports unchanged instead of updated.
retire_converged_result = reconcile_run(confirmed_retire_run)
print(f'Reconcile result on the following sweep (converged): {retire_converged_result}')
assert (
    retire_converged_result.detach_declined == 1
), 'the declined retirement is reported on every sweep, not only the first'
assert retire_converged_result.unchanged == 1, 'a second sweep with nothing else changed reports unchanged, not updated'

Confirmed-retirement demo run: pk=78  window=2026-09-11


No Profile found for reconciler-demo-retirement-confirming-staffer. Creating Profile.


Target post save hook: BTckhijjEcwyZIghmkRa created: True


Target post save hook: BTckhijjEcwyZIghmkRa created: False


Minted night: pk=374  url='ALLOC:78:2026-09-11'
Confirmed by: 'reconciler-demo-retirement-confirming-staffer'


...

Allocation retire declined: night pk=374 night=2026-09-11 is human-confirmed to run pk=78 -- an automated retirement never destroys it.


Allocation retire declined: night pk=374 night=2026-09-11 is human-confirmed to run pk=78 -- an automated retirement never destroys it.


Allocation retire declined: night pk=374 night=2026-09-11 is human-confirmed to run pk=78 -- an automated retirement never destroys it.


Allocation retire declined: night pk=374 night=2026-09-11 is human-confirmed to run pk=78 -- an automated retirement never destroys it.


Linked ObservationRecord pk=224 to confirmed_retire_run pk=78 via CampaignRunObservation pk=3 -- the retirement guard fires here, from the receiver alone, before any explicit reconcile_run() call.
Allocation night pk=374 still exists after the link, with its confirmation intact (confirmed_by='reconciler-demo-retirement-confirming-staffer').
Reconcile result reproducing the same decision: ReconcileResult(created=0, updated=0, unchanged=1, blocked=0, skipped_nights=0, detached=0, detach_declined=1, remint_declined=0, retired=0, rekeyed=0, legacy_deleted=0, skipped_reason=None)
The calendar now shows BOTH the confirmed allocation night AND the linked observation's own entry on the same night -- the duplicate the runbook's detach_declined section describes, resolved only by clearing the confirmation and re-running the sweep.
Title before the cancellation sweep: 'RDGS EFOSC2 (confirmed-retirement demo)'
Reconcile result after marking the run cancelled: ReconcileResult(created=0, updated=1, 

## The `remint_declined` counter in a real summary line (plan 35-23, WR-06)

35-REVIEW.md WR-06: the two operator-facing messages that used to accompany a re-mint
decline both said "left attributed -- someone had already confirmed them", which was false
for a re-mint decline (nothing was "left attributed"; no attribution was at stake). Plan
35-23 split the counter and gave the re-mint decline its own message. The cell below runs
the actual management command (not `reconcile_run()` directly) against the whole scratch
database -- both `declined_run`'s still-declined night above and `confirmed_retire_run`'s
still-declined retirement above are picked up again on this sweep, since neither has been
resolved -- and shows the real `remint_declined: N` token in the final summary line
alongside the real per-run stderr line an operator would actually read.


In [21]:
import io

from django.core.management import call_command

remint_stdout = io.StringIO()
remint_stderr = io.StringIO()
call_command('reconcile_campaign_runs', stdout=remint_stdout, stderr=remint_stderr)

remint_stdout_text = remint_stdout.getvalue()
remint_stderr_text = remint_stderr.getvalue()
print('stdout:', remint_stdout_text)
print('stderr:', remint_stderr_text)

assert 'remint_declined: ' in remint_stdout_text, 'the summary line must carry the remint_declined token'
summary_line = [line for line in remint_stdout_text.splitlines() if line.startswith('Done.')][0]
remint_declined_value = int(summary_line.split('remint_declined: ')[1].split(',')[0])
assert remint_declined_value >= 1, f'expected at least 1 remint_declined, got {remint_declined_value}'
print(f'Real summary line reports remint_declined: {remint_declined_value}')

per_run_remint_lines = [
    line for line in remint_stderr_text.splitlines() if 'kept' in line and 'existing boundaries' in line
]
assert per_run_remint_lines, 'expected at least one per-run remint_declined stderr line'
print('Per-run stderr line:', per_run_remint_lines[0])

Allocation re-mint declined: event pk=373 night=2026-09-10 is human-confirmed to run pk=77 -- an automated re-mint never clears it.


Allocation retire declined: night pk=374 night=2026-09-11 is human-confirmed to run pk=78 -- an automated retirement never destroys it.


stdout: Done. runs: 54, created: 0, updated: 0, unchanged: 81, skipped: 9, failed: 0, blocked: 0, skipped_nights: 0, detached: 0, detach_declined: 1, remint_declined: 1, retired: 0, rekeyed: 0, legacy_deleted: 0

stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)
Run pk=77: 1 allocation night kept its existing boundaries -- a person's confirmation, an observation link, or an unverified companion row outranks this automated correction; see the runbook's remint_declined section for the remedy
Run pk=78: 1 superseded entry left attributed -- a person confirmed them, and an automated sweep never clears a human confirmation

Real summary line reports remint_declined: 1
Per-run stderr line: Run pk=77: 1 allocation night kept its existing bou

## An in-place site-definition correction re-mints (plan 35-24, the escalated decision)

35-VERIFICATION.md's "Human Verification Required" #1, escalated after the round-5
verifier's own probe: before plan 35-24, the recorded mint-provenance token carried only
`site_id` (added by plan 35-21's CR-02), never the site's own POSITION. Correcting
`CampaignRun.site` to a different `Observatory` row re-minted correctly (demonstrated in
the site-correction cell above), but editing the SAME `Observatory` row's `lat`/`lon`/
`altitude`/`timezone` in place -- never touching any run's `site` at all -- compared the
unchanged `site_id` equal to itself and reported `unchanged` forever. A real position
correction (a mis-entered coordinate, for example) was silently invisible: every night
already projected at that site kept its stale boundary, wrong by up to fifteen hours.

The `v3` token now also carries a 16-character fingerprint of the site's own position and
timezone, so an in-place correction is detected the same way a `site` reassignment already
was. The cell below creates a dedicated `Observatory` and a dedicated single-night run (so
this demo does not disturb any site the cells above already used), mints the night, THEN
edits the SAME `Observatory` row's position in place -- reproducing the round-5 verifier's
own probe exactly -- and reconciles again.


In [22]:
from solsys_code.allocation_projector import night_bounds
from solsys_code.telescope_runs import sun_event
from solsys_code.solsys_code_observatory.models import Observatory

escalated_site, _ = Observatory.objects.update_or_create(
    obscode='X32',
    defaults=dict(
        name='Reconciler Demo Site-Definition-Correction Site',
        short_name='RDSC',
        lat=-29.0000,
        lon=-70.0000,
        altitude=2000,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(
    f'Site before correction: obscode={escalated_site.obscode!r}  lat={escalated_site.lat}  lon={escalated_site.lon}  altitude={escalated_site.altitude}'
)

escalated_run, _ = CampaignRun.objects.update_or_create(
    campaign=None,
    telescope_instrument='RDSC/EFOSC2 (site-definition-correction demo)',
    window_start=date(2026, 9, 12),
    window_end=date(2026, 9, 12),
    defaults=dict(
        site=escalated_site,
        site_raw='X32',
        observation_details='Single-night run for the site-definition-correction demo',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
    ),
)
print(f'Site-definition-correction demo run: pk={escalated_run.pk}  window={escalated_run.window_start}')

escalated_night = escalated_run.window_start
escalated_url = allocation_night_url(escalated_run, escalated_night)

first_result = reconcile_run(escalated_run)
print(f'First reconcile (mints the night): {first_result}')

escalated_event_before = CalendarEvent.objects.get(url=escalated_url)
escalated_pk_before = escalated_event_before.pk
escalated_start_before = escalated_event_before.start_time
escalated_end_before = escalated_event_before.end_time
print(
    f'Before site-definition correction: pk={escalated_pk_before}  '
    f'start={escalated_start_before.isoformat()}  end={escalated_end_before.isoformat()}'
)

# A second reconcile with NOTHING changed must be a no-op -- the control this demo needs
# before correcting the site, so the re-mint below is attributable to the correction alone.
idempotent_result = reconcile_run(escalated_run)
assert idempotent_result.retired == 0 and idempotent_result.created == 0
print(f'Idempotent reconcile with nothing changed: {idempotent_result}')

# Correct the SAME Observatory row's position in place -- escalated_run.site is never
# reassigned, and escalated_run itself is never saved.
escalated_site.lat = -29.2567
escalated_site.lon = -70.7300
escalated_site.altitude = 2347
escalated_site.save(update_fields=['lat', 'lon', 'altitude'])
print(
    f'Site after correction:  obscode={escalated_site.obscode!r}  lat={escalated_site.lat}  lon={escalated_site.lon}  altitude={escalated_site.altitude}'
)

correction_result = reconcile_run(escalated_run)
print(f'Reconcile result after the in-place site-definition correction: {correction_result}')

escalated_event_after = CalendarEvent.objects.get(url=escalated_url)
expected_sunset, expected_sunrise = sun_event(escalated_site, escalated_night, kind='sun')
expected_start, expected_end = night_bounds(escalated_run, escalated_night, expected_sunset, expected_sunrise)
print(
    f'After site-definition correction:  pk={escalated_event_after.pk}  '
    f'start={escalated_event_after.start_time.isoformat()}  end={escalated_event_after.end_time.isoformat()}'
)
print(
    "Corrected site's own sun_event() boundary:      "
    f'start={expected_start.isoformat()}  end={expected_end.isoformat()}'
)

assert correction_result.retired == 1, 'an in-place site-definition correction must re-mint (retired==1)'
assert correction_result.created == 1, 'an in-place site-definition correction must re-mint (created==1)'
assert (
    escalated_event_after.pk != escalated_pk_before
), 'a site-definition correction must re-mint (a new primary key), not report unchanged forever'
assert escalated_event_after.start_time != escalated_start_before, 'the boundary must actually move'
assert escalated_event_after.start_time == expected_start
assert escalated_event_after.end_time == expected_end
print(
    "Boundaries moved to the corrected site's real sun_event() values, without escalated_run.site ever "
    'being reassigned -- before plan 35-24 this correction was invisible and the night would have kept '
    'reporting unchanged forever, silently wrong (the escalated decision closed).'
)

Site before correction: obscode='X32'  lat=-29.0  lon=-70.0  altitude=2000
Site-definition-correction demo run: pk=79  window=2026-09-12


First reconcile (mints the night): ReconcileResult(created=1, updated=0, unchanged=0, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=0, rekeyed=0, legacy_deleted=0, skipped_reason=None)
Before site-definition correction: pk=375  start=2026-09-12T22:37:18+00:00  end=2026-09-13T10:34:20+00:00
Idempotent reconcile with nothing changed: ReconcileResult(created=0, updated=0, unchanged=1, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=0, rekeyed=0, legacy_deleted=0, skipped_reason=None)
Site after correction:  obscode='X32'  lat=-29.2567  lon=-70.73  altitude=2347


Allocation unrecorded-provenance night pk=375 run pk=79 night=2026-09-12: stored boundary start=2026-09-12 22:37:18+00:00 end=2026-09-13 10:34:20+00:00 disagrees beyond tolerance with the resolved sun event sunset=2026-09-12 22:40:39+00:00 sunrise=2026-09-13 10:36:49+00:00.


Reconcile result after the in-place site-definition correction: ReconcileResult(created=1, updated=0, unchanged=0, blocked=0, skipped_nights=0, detached=0, detach_declined=0, remint_declined=0, retired=1, rekeyed=0, legacy_deleted=0, skipped_reason=None)


After site-definition correction:  pk=376  start=2026-09-12T22:40:39+00:00  end=2026-09-13T10:36:49+00:00
Corrected site's own sun_event() boundary:      start=2026-09-12T22:40:39+00:00  end=2026-09-13T10:36:49+00:00
Boundaries moved to the corrected site's real sun_event() values, without escalated_run.site ever being reassigned -- before plan 35-24 this correction was invisible and the night would have kept reporting unchanged forever, silently wrong (the escalated decision closed).


## Summary

`reconcile_campaign_runs` is for sweeps and backfills, not routine use: `approve()`,
`_resolve_site()`, `mark_cancelled` and `mark_weather_failure` in `campaign_views.py` each
call `campaign_reconciler.reconcile_run()` directly, so a single run's calendar events are
reconciled immediately the moment a staff member takes one of those actions on it -- no
command run is needed for that common case.

This notebook is the executed proof of ROADMAP Success Criterion 5: the classical cutover
section above converts the developer database's legacy blank-url classical events and its
retired `RUN:{pk}:{date}` per-night family into the current `ALLOC:` allocation layer, with
four end-state properties asserted in code (zero date-bearing nights, only the reported
unexplained blank-url row(s) remain, the containers unchanged in count, and every
facility-url observation event byte-identical). The dispatch-branch demo shows all four of
`reconcile_run()`'s current branches (class-wide, satellite, queue, and campaign-less
classical), the observation-handoff demo shows the D-05/D-07 property that retires and
restores an allocation night as a real observation links and unlinks, and the
site-correction demo shows plan 35-21's CR-02 fix re-minting an already-projected run's
nights to a corrected site's real sun-event boundaries instead of reporting `unchanged`
forever.

Four further demos prove this round's (plans 35-23 and 35-24) behaviour changes with real
executed output:

- **A declined re-mint still refreshes the night's labelling, and survives its own
  retirement** (plan 35-23, CR-01/CR-04): a declined boundary rewrite is reported under the
  new `remint_declined` counter (not `detach_declined`), while the SAME sweep still lands
  the `[C]` marker on the title via the ordinary plain-update path -- the primary
  key, both boundaries and the confirmation stamp all survive unchanged.
- **A confirmed allocation night survives its own retirement** (plan 35-23, CR-05): linking
  an `ObservationRecord` whose block is already placed on a confirmed night no longer
  deletes that night -- the retirement guard fires from the `CampaignRunObservation`
  `post_save` receiver alone, reported under `detach_declined`, leaving the calendar with
  both the confirmed allocation night and the observation's own entry until the confirmation
  is cleared.
- **The `remint_declined` counter in a real summary line** (plan 35-23, WR-06): the actual
  `reconcile_campaign_runs` management command -- not `reconcile_run()` called directly --
  reports the new counter in its final summary line and prints the real per-run stderr
  message an operator would read.
- **An in-place site-definition correction re-mints** (plan 35-24, the escalated decision):
  editing an `Observatory` row's own position in place, without ever reassigning any run's
  `site`, now re-mints every allocation night already projected at that site -- before this
  release such a correction was invisible, and a night would have kept reporting `unchanged`
  forever, silently wrong by up to fifteen hours.

See `docs/runbooks/telescope_runs_calendar.rst`, "How do I run the one-time classical
cutover?" and "How do I get every campaign run onto the calendar?", for the operator-facing
version of everything demonstrated in this notebook.


## Scratch database teardown

Removes the scratch copy created in the setup cell above. The developer database
was never opened for writing by this notebook run.

In [23]:
import shutil

shutil.rmtree(scratch_db_dir, ignore_errors=True)
print(f'Removed scratch database directory: {scratch_db_dir}')
print('The developer database (src/fomo_db.sqlite3) was never opened for writing.')

Removed scratch database directory: /tmp/fomo-notebook-db-ycs_q23l
The developer database (src/fomo_db.sqlite3) was never opened for writing.
